In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
pip install tf-keras -q

In [ ]:
!pip install scikit-image plotly 



In [ ]:
pip install "transformers<5.0.0"

In [ ]:
import os
# This MUST be set before importing tensorflow or transformers
os.environ["TF_USE_LEGACY_KERAS"] = "1"

from transformers import TFAutoModel
import tensorflow as tf

# Now it will load perfectly without the ImportError
model = TFAutoModel.from_pretrained("google/vit-base-patch16-224-in21k",
use_safetensors=False
)

In [ ]:
!pip install open3d -q
import os
os.environ['OPEN3D_CPU_RENDERING']='true'
import open3d as o3d 
import numpy as np
import torch

In [ ]:
pip install transformers -q


In [ ]:
pip install "transformers<5.0.0"


In [ ]:
def build_vit_encoder():
    vit = TFAutoModel.from_pretrained("google/vit-base-patch16-224-in21k", use_safetensors=False)
    inputs=tf.keras.Input(shape=(224, 224, 3), name="image_input")
    x=(inputs/127.5)-1.0
    x=tf.transpose(x, perm=[0,3,1,2])
    outputs=vit(pixel_values=x)
    patch_features=outputs.last_hidden_state[:, 1:, :]
    return tf.keras.Model(inputs=inputs, outputs=patch_features, name="vit_encoder")

class VoxelCrossAttentionDecoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim=768, num_heads=4):
        super().__init__()
        self.cross_attn=tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.norm1=tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2=tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ffn=tf.keras.Sequential([
            tf.keras.layers.Dense(embed_dim*4, activation='relu'),
            tf.keras.layers.Dense(embed_dim),
        ])

    def call(self, voxel_queries, image_features):
        attn_out=self.cross_attn(query=voxel_queries, value=image_features, key=image_features)
        x=self.norm1(voxel_queries+attn_out)
    
        ffn_out=self.ffn(x)
        return self.norm2(x+ffn_out)

def build_3d_pipeline(grid_size=16, embed_dim=768):
    num_voxels=grid_size**3

    image_input=tf.keras.Input(shape=(224, 224, 3), name="image_input")
    # --- NEW: Create a 3D coordinate grid (X, Y, Z) ---
    coords = np.stack(np.meshgrid(
        np.linspace(-1, 1, grid_size),
        np.linspace(-1, 1, grid_size),
        np.linspace(-1, 1, grid_size)
    ), axis=-1).reshape(-1, 3) # Shape: (num_voxels, 3)
    
    # Project 3D coordinates to the embedding dimension
    coord_init = tf.keras.layers.Dense(embed_dim)(coords.astype(np.float32))
    
    # Make these coordinates learnable queries
    voxel_query_embeddings = tf.Variable(
        initial_value=tf.expand_dims(coord_init, 0), 
        trainable=True,
        name="positioned_voxels"
    )

    vit_encoder=build_vit_encoder()
    image_features=vit_encoder(image_input)
    batch_size=tf.shape(image_input)[0]
    batched_voxel_queries=tf.tile(voxel_query_embeddings, [batch_size,1, 1])

    decoder_block=VoxelCrossAttentionDecoder(embed_dim=embed_dim)
    decoder_voxels=decoder_block(batched_voxel_queries, image_features)

    x=tf.keras.layers.Dense(128, activation='relu')(decoder_voxels)
    tsdf_values=tf.keras.layers.Dense(1, activation='linear')(x)

    output_grid=tf.keras.layers.Reshape((grid_size, grid_size, grid_size))(tsdf_values)

    return tf.keras.Model(inputs=image_input, outputs=output_grid, name="Full_3D_Model")

print("Architecture defined. ")


    

In [ ]:
import numpy as np 
from skimage import measure 
import plotly.figure_factory as ff


In [ ]:
print("downloading weights and building model graph")
model=build_3d_pipeline(grid_size=32, embed_dim=768)
dummy_image=np.random.rand(1,224, 224, 3).astype(np.float32)*255.0
print("Predicting 3D Geometry")
predicted_tsdf=model(dummy_image, training=False)
tsdf_grid=predicted_tsdf[0].numpy()

print("marching cubes")
fallback_level=np.median(tsdf_grid)
verts, faces, normals, values=measure.marching_cubes(tsdf_grid, level=fallback_level)
print(f"Rendering {len(verts)} vertices in 3D")

fig=ff.create_trisurf(
    x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
    simplices=faces,
    title="Untrained Vox Model Output",
    aspectratio=dict(x=1, y=1, z=1)
)
fig.show(renderer='iframe')


In [ ]:
def surface_weighted_l1_loss(y_true, y_pred, tau=0.18):
    abs_error=tf.abs(y_true-y_pred)
    surface_mask=tf.where(tf.abs(y_true)<tau, tf.ones_like(y_true), tf.fill(tf.shape(y_true), 0.18))
    return tf.reduce_mean(abs_error*surface_mask)

def eikonal_loss(y_pred, tau=0.18, grid_size=16):
    h = 1.0 / grid_size

    grad_x = (y_pred[:, 1:, :-1, :-1] - y_pred[:, :-1, :-1, :-1])/h
    grad_y = (y_pred[:, :-1, 1:, :-1] - y_pred[:, :-1, :-1, :-1])/h
    grad_z = (y_pred[:, :-1, :-1, 1:] - y_pred[:, :-1, :-1, :-1])/h

    grad_norm=tf.sqrt(tf.square(grad_x)+tf.square(grad_y)+tf.square(grad_z)+ 1e-8)
    y_pred_cropped = y_pred[:, :-1, :-1, :-1]
    mask = tf.cast(tf.abs(y_pred_cropped) < tau, tf.float32)

    error= tf.reduce_mean(tf.square(grad_norm-1.0))*mask
    return tf.reduce_sum(error) / (tf.reduce_sum(mask) + 1e-8)

In [ ]:
pip install glob


In [ ]:
import os
import tensorflow as tf
import numpy as np

# 1. The Real Paths from your diagnostic
img_folder = '/kaggle/input/datasets/aahannayak/images/images'
vox_folder = '/kaggle/input/datasets/aahannayak/voxels/voxels'

# 2. Grab filenames directly using os.listdir
# This is more robust than glob on Kaggle
img_files = sorted([f for f in os.listdir(img_folder) if f.endswith(('.jpg', '.png'))])
vox_files = sorted([f for f in os.listdir(vox_folder) if f.endswith('.npy')])

# 3. Join them to make full absolute paths
list_of_img_paths = [os.path.join(img_folder, f) for f in img_files]
list_of_vox_paths = [os.path.join(vox_folder, f) for f in vox_files]

print(f"✅ FOUND: {len(list_of_img_paths)} images")
print(f"✅ FOUND: {len(list_of_vox_paths)} voxels")

# 4. SAFETY CHECK: Match the counts
# Sometimes Kaggle adds extra system files; let's make sure they are equal
min_count = min(len(list_of_img_paths), len(list_of_vox_paths))
list_of_img_paths = list_of_img_paths[:min_count]
list_of_vox_paths = list_of_vox_paths[:min_count]

if min_count == 0:
    raise ValueError(f"🚨 Still zero! Check this folder manually: {img_folder}")

# 5. Build Dataset
def load_perfect_data(img_path, vox_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [224, 224])
    img = (tf.cast(img, tf.float32) / 127.5) - 1.0
    
    def load_npy(path):
        # We need to use decode() because TF strings are bytes
        return np.load(path.decode('utf-8')).astype(np.float32)
    
    tsdf = tf.numpy_function(load_npy, [vox_path], tf.float32)
    tsdf.set_shape((32, 32, 32)) 
    return img, tsdf

batch_size = 2
dataset = tf.data.Dataset.from_tensor_slices((list_of_img_paths, list_of_vox_paths))
dataset = dataset.shuffle(1000).map(load_perfect_data, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

print("🚀 DATASET READY. Start training!")

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
epochs = 50

# The Lambda weight controls how strongly the Eikonal loss affects the model.
# Too high, and it ignores the target shape. Too low, and the surface gets bumpy.
eikonal_weight = 0.1

@tf.function
def train_step(images, ground_truth_tsdf):
    with tf.GradientTape() as tape:
        # 1. Forward Pass
        predicted_tsdf = model(images, training=True)
        
        # 2. Calculate L1 Shape Loss
        shape_loss = surface_weighted_l1_loss(ground_truth_tsdf, predicted_tsdf, tau=0.1)
        
        # 3. Calculate Eikonal Constraint Loss
        eik_loss = eikonal_loss(predicted_tsdf, tau=0.1)
        
        # 4. Total Combined Loss
        total_loss = shape_loss + (eikonal_weight * eik_loss)
        
    gradients = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    return total_loss, shape_loss, eik_loss

print("Starting Eikonal-Constrained Training...")


print(f"Model initialized! Trainable variables created: {len(model.trainable_variables)}")
for epoch in range(epochs):
    for batch_idx, (batch_images, batch_tsdf) in enumerate(dataset):
        
        t_loss, s_loss, e_loss = train_step(batch_images, batch_tsdf)
        
        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}")
            print(f"Total: {t_loss:.4f} | Shape: {s_loss:.4f} | Eikonal: {e_loss:.4f}")
            print("-" * 40)

In [ ]:
import os

# This looks at everything in your Kaggle input folder
for root, dirs, files in os.walk('/kaggle/input'):
    if 'images' in dirs and 'voxels' in dirs:
        print(f"🎯 FOUND IT! Your images are in: {os.path.join(root, 'images')}")
        print(f"🎯 FOUND IT! Your voxels are in: {os.path.join(root, 'voxels')}")
        break
else:
    print("❌ Still can't find folders named 'images' and 'voxels'.")

In [ ]:
import matplotlib.pyplot as plt 
import numpy as np



def test_reconstruction(image_path, model, grid_size=16):
    img=tf.io.read_file(image_path)
    img=tf.image.decode_jpeg(img, channels=3)
    img=tf.image.resize(img, [224, 224])
    input_tensor=tf.expand_dims(img, axis=0)
    
    predicted_grid=model(input_tensor, training=False)
    predicted_grid=predicted_grid.numpy()[0]
    
    fig, axes=plt.subplots(1, 3, figsize=(15,5))
    axes[0].imshow(img.numpy().astype("uint8"))
    axes[0].set_title("Input Image")
    axes[0].axis('off')
    
    mid_slice = predicted_grid[grid_size//2, :, :]
    im1 = axes[1].imshow(mid_slice, cmap='seismic', vmin=-0.1, vmax=0.1)
    axes[1].set_title(f"Horizontal Slice (Z={grid_size//2})")
    fig.colorbar(im1, ax=axes[1], shrink=0.6)
    
    vert_slice = predicted_grid[:, grid_size//2, :]
    im2 = axes[2].imshow(vert_slice, cmap='seismic', vmin=-0.1, vmax=0.1)
    axes[2].set_title(f"Vertical Slice (Y={grid_size//2})")
    fig.colorbar(im2, ax=axes[2], shrink=0.6)

  
    
    
    plt.tight_layout()
    plt.show()
    
    return predicted_grid




In [ ]:
import tensorflow as tf
image_path = "/kaggle/input/datasets/aahannayak/images/images/shape_000000.jpg" 
res = test_reconstruction(image_path, model)

In [ ]:
print(f"Max value: {res.max()}")
print(f"Min value: {res.min()}")
print(f"Average value: {res.mean()}")

In [ ]:
import scipy.ndimage as ndimage

# Smooth the grid to remove the "static"
smoothed_grid = ndimage.gaussian_filter(res, sigma=0.3)

# Now plot the smoothed version
plt.imshow(smoothed_grid[8, :, :], cmap='seismic', vmin=-0.05, vmax=0.05)
plt.title("Smoothed Horizontal Slice")
plt.show()

In [ ]:
from skimage import measure

# Find the surface where TSDF = 0
verts, faces, normals, values = measure.marching_cubes(res, level=0.0)

# This will tell you how many 'triangles' your car has
print(f"Mesh has {len(faces)} triangles.")

In [ ]:
from skimage import measure

# Extract the car surface
verts, faces, normals, values = measure.marching_cubes(res, level=0.0)

# Save as a standard .obj file
with open("reconstructed_car.obj", "w") as f:
    for v in verts:
        f.write(f"v {v[0]} {v[1]} {v[2]}\n")
    for face in faces:
        # obj indices start at 1
        f.write(f"f {face[0]+1} {face[1]+1} {face[2]+1}\n")

print("Finished! Download 'reconstructed_car.obj' from your Kaggle output folder.")

In [ ]:
model.save_weights('m4_reconstructor_weights.h5')